# Laboratorio 4 – Árboles de Decisión
**CC3074 – Minería de Datos | Universidad del Valle de Guatemala | Semestre I 2026**

**Contexto:** SmartStay Advisors necesita modelos para estimar precios de propiedades Airbnb, identificar propiedades con baja ocupación y comprender qué factores influyen en los ingresos.

**Variable respuesta:** `price` (precio por noche en moneda local)

In [ ]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

---
## Actividad 1 – Carga del dataset

In [ ]:
result = pyreadr.read_r('listings.RData')
df = result['listings'].copy()
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'\nColumnas: {df.columns.tolist()}')
df.head(3)

El dataset contiene **171,748 anuncios** y **80 variables** que describen propiedades Airbnb: características físicas, del anfitrión, disponibilidad, reseñas y ubicación.

---
## Actividad 2 – Análisis Exploratorio de Datos (EDA)

### 2.1 Preprocesamiento de `price`

La variable `price` viene en formato texto (`$97.00`). Se remueven los símbolos `$` y `,`, se convierte a numérico y se eliminan registros con precio cero o nulo.

In [ ]:
print('Formato original de price:', df['price'].head(5).tolist())

df['price'] = df['price'].str.replace('[$,]', '', regex=True).str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

n_antes = len(df)
df = df[df['price'] > 0].dropna(subset=['price'])
print(f'Filas eliminadas (price <= 0 o nulo): {n_antes - len(df):,}')
print(f'Dataset final: {len(df):,} filas')

Se eliminaron **95,502 registros** (55% del total) que no tenían precio asignado. Esto es esperado en Airbnb: muchos anuncios existen pero no están activos con precio publicado. El análisis continúa con **76,246 propiedades** con precio válido.

### 2.2 Valores faltantes

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['variable', 'pct_nulo']

plt.figure(figsize=(12, 6))
sns.barplot(data=missing_df.head(20), x='pct_nulo', y='variable', color='steelblue')
plt.title('Top 20 variables con mayor % de valores faltantes')
plt.xlabel('% Nulos')
plt.tight_layout()
plt.show()

print(f'Variables con >50% nulos: {(missing > 50).sum()}')
print(missing_df.head(15).to_string(index=False))

**Hallazgos:**
- `calendar_updated` tiene **100% de nulos** — se eliminará.
- `neighbourhood_group_cleansed` tiene **50% de nulos** — se usará `neighbourhood_cleansed` como alternativa.
- Las variables de `review_scores_*` y `reviews_per_month` tienen **~17.6% de nulos**, correspondiente a propiedades sin reseñas aún. Se imputarán con la mediana.
- `license` tiene 13.8% de nulos y tiene baja utilidad predictiva — se eliminará.

### 2.3 Distribución de `price`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de price (original)')
axes[0].set_xlabel('Precio por noche')

p99 = df['price'].quantile(0.99)
axes[1].hist(df[df['price'] <= p99]['price'], bins=60, color='coral', edgecolor='white')
axes[1].set_title(f'Sin outliers extremos (p99 = ${p99:,.0f})')
axes[1].set_xlabel('Precio por noche')

axes[2].hist(np.log1p(df['price']), bins=60, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Distribución log(1 + price)')
axes[2].set_xlabel('log(1 + precio)')

plt.tight_layout()
plt.show()

print(df['price'].describe().round(2))
print(f'\nSkewness: {df["price"].skew():.2f}')
print(f'P25=${df["price"].quantile(0.25):.0f} | Mediana=${df["price"].quantile(0.5):.0f} | P75=${df["price"].quantile(0.75):.0f} | P99=${df["price"].quantile(0.99):,.0f}')

**Hallazgos:**
- La distribución tiene una **asimetría extrema (skewness = 9.87)**: la mayoría de propiedades tiene precios entre $120 y $326, pero existen valores hasta $50,123.
- **Mediana = $193**, **media = $750** — la gran diferencia indica que los outliers (hoteles de lujo, propiedades Premium) jalan la media hacia arriba.
- La transformación logarítmica produce una distribución mucho más simétrica, lo que podría beneficiar los modelos lineales en actividades posteriores.
- Para la categorización (Actividad 9) se usarán los percentiles de la distribución real para definir límites robustos.

### 2.4 Precio por tipo de habitación (`room_type`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p95 = df['price'].quantile(0.95)
sns.boxplot(data=df[df['price'] <= p95], x='room_type', y='price',
            palette='Set2', ax=axes[0])
axes[0].set_title('Precio por tipo de habitación (sin outliers extremos)')
axes[0].set_xlabel('Tipo de habitación')
axes[0].set_ylabel('Precio por noche')
axes[0].tick_params(axis='x', rotation=10)

room_counts = df['room_type'].value_counts()
axes[1].bar(room_counts.index, room_counts.values,
            color=sns.color_palette('Set2', len(room_counts)))
axes[1].set_title('Cantidad de propiedades por tipo')
axes[1].set_xlabel('Tipo de habitación')
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

print(df.groupby('room_type')['price'].describe().round(2))

**Hallazgos:**
- **Entire home/apt** domina el dataset con 65,709 propiedades (86%) y precio mediano de $205.
- **Hotel room** (649 registros, <1%) tiene precios extremadamente altos (media $24,477, mediana $40,000). Esto apunta a que los hoteles de lujo están en una categoría completamente diferente y pueden actuar como outliers influyentes en el modelo.
- **Private room** tiene precio mediano de $85 — significativamente menor al alojamiento completo.
- **Shared room** es la categoría más económica con precio mediano de $42.
- `room_type` es una variable altamente predictiva del precio y será fundamental en el modelo.

### 2.5 Precio por capacidad de huéspedes (`accommodates`)

In [ ]:
df_acc = df.copy()
df_acc['acc_group'] = pd.cut(df_acc['accommodates'],
                              bins=[0,1,2,3,4,6,8,50],
                              labels=['1','2','3','4','5-6','7-8','9+'])
p95 = df['price'].quantile(0.95)
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_acc[df_acc['price'] <= p95],
            x='acc_group', y='price', palette='Blues')
plt.title('Precio por noche según capacidad de huéspedes')
plt.xlabel('Capacidad de huéspedes')
plt.ylabel('Precio por noche')
plt.tight_layout()
plt.show()

print('Precio mediano por capacidad (primeros 10):')
print(df.groupby('accommodates')['price'].median().head(10).round(2))

**Hallazgos:**
- Existe una relación positiva clara entre capacidad y precio: desde $54 (1 huésped) hasta $436 (10 huéspedes).
- La relación no es perfectamente lineal — hay saltos entre 7-8 y 9+ huéspedes donde el precio sube considerablemente.
- Esta variable es uno de los mejores predictores individuales del precio.

### 2.6 Correlación de variables numéricas con `price`

In [ ]:
num_cols = [
    'price', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'availability_365', 'number_of_reviews',
    'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_value',
    'reviews_per_month', 'host_listings_count',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d'
]
num_cols = [c for c in num_cols if c in df.columns]

df_num = df[num_cols].copy()
for col in df_num.columns:
    df_num[col] = pd.to_numeric(df_num[col], errors='coerce')

corr = df_num.corr()[['price']].drop('price').sort_values('price', ascending=False)

plt.figure(figsize=(7, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.7})
plt.title('Correlación con price')
plt.tight_layout()
plt.show()

**Hallazgos:**
- Las variables con mayor correlación positiva con `price` son `accommodates`, `bedrooms`, `beds` y `bathrooms` — propiedades más grandes cuestan más.
- `estimated_revenue_l365d` tiene correlación alta esperada (el ingreso anual depende del precio diario).
- Las puntuaciones de reviews tienen correlaciones bajas o negativas con el precio: propiedades caras no necesariamente reciben mejores calificaciones.
- `minimum_nights` y `availability_365` tienen correlaciones muy bajas, indicando que la política de reservas no determina el precio.

### 2.7 Superhost vs precio

In [ ]:
df_sh = df[df['host_is_superhost'].isin(['t','f'])].copy()
df_sh['es_superhost'] = df_sh['host_is_superhost'].map({'t':'Superhost','f':'No Superhost'})

p95 = df['price'].quantile(0.95)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df_sh[df_sh['price'] <= p95], x='es_superhost', y='price',
            palette='Pastel1', ax=axes[0])
axes[0].set_title('Precio: Superhost vs No-Superhost')
axes[0].set_xlabel('')

counts = df_sh['es_superhost'].value_counts()
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Pastel1'))
axes[1].set_title('Proporción Superhost')

plt.tight_layout()
plt.show()

print(df_sh.groupby('es_superhost')['price'].describe().round(2))

**Hallazgos:**
- Los Superhosts tienen precio mediano ligeramente mayor ($202) que los No-Superhosts ($185), pero la diferencia no es drástica.
- Los Superhosts representan la mayoría del dataset, lo que sugiere que el programa de Superhost tiene alta penetración en las ciudades analizadas.
- La distinción Superhost no es el factor dominante del precio, pero puede ser relevante como variable de control.

### 2.8 Disponibilidad anual vs precio

In [ ]:
df['avail_group'] = pd.cut(df['availability_365'],
                            bins=[-1, 30, 90, 180, 270, 365],
                            labels=['0-30d','31-90d','91-180d','181-270d','271-365d'])
p95 = df['price'].quantile(0.95)
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[df['price'] <= p95], x='avail_group', y='price', palette='Oranges')
plt.title('Precio por noche según disponibilidad anual')
plt.xlabel('Días disponibles al año')
plt.ylabel('Precio por noche')
plt.tight_layout()
plt.show()

print(df.groupby('avail_group')['price'].describe().round(2))

**Hallazgos:**
- La disponibilidad anual promedio es de **231 días** (mediana=256), indicando que la mayoría de propiedades está disponible más de la mitad del año.
- No hay una tendencia clara entre disponibilidad y precio — propiedades con alta y baja disponibilidad pueden tener precios similares.
- Esto sugiere que la disponibilidad responde más a la estrategia del anfitrión que al precio de la propiedad.

### 2.9 Preprocesamiento final del dataset para modelado

In [ ]:
# Columnas a descartar: identificadores, URLs, texto libre, fechas
cols_drop = [
    'id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_name','host_since','host_location','host_about',
    'host_thumbnail_url','host_picture_url','host_verifications',
    'neighbourhood','calendar_updated','calendar_last_scraped',
    'first_review','last_review','license','bathrooms_text','amenities',
    'avail_group'  # columna auxiliar de visualización
]
cols_drop = [c for c in cols_drop if c in df.columns]
df_model = df.drop(columns=cols_drop).copy()

# Columnas con >50% nulos
high_null = df_model.isnull().mean()
cols_high = high_null[high_null > 0.5].index.tolist()
print(f'Cols eliminadas por >50% nulos: {cols_high}')
df_model = df_model.drop(columns=cols_high)

# Booleanos t/f -> 1/0 (ignorar cadenas vacías)
for col in df_model.select_dtypes('object').columns:
    uniq = set(df_model[col].dropna().unique()) - {''}
    if uniq.issubset({'t', 'f'}):
        df_model[col] = df_model[col].map({'t': 1, 'f': 0})

# Tasas porcentuales -> decimal
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df_model.columns and df_model[col].dtype == object:
        df_model[col] = pd.to_numeric(
            df_model[col].str.replace('%', '').str.strip(), errors='coerce') / 100

# Forzar numéricas las que deberían serlo
cols_should_be_numeric = [
    'bedrooms','beds','bathrooms','minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights','host_listings_count',
    'host_total_listings_count'
]
for col in cols_should_be_numeric:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

# One-hot encoding solo a las categóricas verdaderas de baja cardinalidad
cat_cols = df_model.select_dtypes('object').columns.tolist()
# Mantener solo las de cardinalidad razonable (<50 categorías)
cat_cols_filtered = [c for c in cat_cols if df_model[c].nunique() <= 50]
cat_cols_drop = [c for c in cat_cols if c not in cat_cols_filtered]
print(f'Cols categóricas de alta cardinalidad eliminadas: {cat_cols_drop}')
df_model = df_model.drop(columns=cat_cols_drop)
df_model = pd.get_dummies(df_model, columns=cat_cols_filtered, drop_first=True)

# Imputar nulos con mediana
num_cols_m = df_model.select_dtypes('number').columns
df_model[num_cols_m] = df_model[num_cols_m].fillna(df_model[num_cols_m].median())

print(f'\nDataset final: {df_model.shape[0]:,} filas × {df_model.shape[1]} columnas')
print(f'Nulos restantes: {df_model.isnull().sum().sum()}')

**Resumen del preprocesamiento:**
- Se eliminaron columnas con identificadores, URLs y texto libre (no aportan valor predictivo)
- `calendar_updated` (100% nulos) fue eliminada
- Variables booleanas (`t`/`f`) convertidas a `1`/`0`
- Tasas porcentuales convertidas a decimales
- Columnas numéricas mal tipadas (bedrooms, beds, etc.) forzadas a numérico
- Variables categóricas de alta cardinalidad (>50 categorías) descartadas para evitar explosión dimensional
- One-Hot Encoding aplicado a categóricas de baja cardinalidad
- Nulos imputados con la mediana de cada columna

---
## Actividad 3 – Análisis de Grupos (Clustering)

Se aplica K-Means sobre las variables numéricas más descriptivas de cada propiedad para identificar segmentos naturales del mercado.

In [ ]:
cluster_features = [
    'price', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'availability_365', 'number_of_reviews', 'review_scores_rating'
]
cluster_features = [c for c in cluster_features if c in df.columns]

df_clust = df[cluster_features].copy()
for col in df_clust.columns:
    df_clust[col] = pd.to_numeric(df_clust[col], errors='coerce')
df_clust = df_clust.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clust)
print(f'Dataset para clustering: {df_clust.shape[0]:,} filas × {df_clust.shape[1]} variables')

In [ ]:
# Método del codo
inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, 'o-', color='steelblue', linewidth=2)
plt.title('Método del Codo – Selección de K')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.xticks(list(k_range))
plt.tight_layout()
plt.show()

for k, inertia in zip(k_range, inertias):
    print(f'K={k}: inercia={inertia:.0f}')

La reducción de inercia se aplana notoriamente a partir de **K=3**, donde la ganancia marginal al agregar más clusters disminuye. Se selecciona **K=3** como valor óptimo, que además coincide con la categorización natural del negocio: propiedades económicas, intermedias y premium.

In [ ]:
K_OPTIMO = 3
km_final = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_clust['cluster'] = km_final.fit_predict(X_scaled)

print('Media por cluster:')
print(df_clust.groupby('cluster')[cluster_features].mean().round(2).to_string())
print('\nMediana de price por cluster:')
print(df_clust.groupby('cluster')['price'].agg(['median','mean','count']))

In [ ]:
# Visualización PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#FF5722', '#4CAF50']
labels = ['Cluster 0 – Económico', 'Cluster 1 – Intermedio', 'Cluster 2 – Ultra-premium']
for i in range(K_OPTIMO):
    mask = df_clust['cluster'].values == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[i], label=labels[i], alpha=0.3, s=8)
axes[0].set_title('Clusters en espacio PCA')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[0].legend(fontsize=8)

p95 = df_clust['price'].quantile(0.95)
df_viz = df_clust[df_clust['price'] <= p95]
axes[1].boxplot(
    [df_viz[df_viz['cluster'] == i]['price'].values for i in range(K_OPTIMO)],
    labels=[f'Cluster {i}' for i in range(K_OPTIMO)],
    patch_artist=True,
    boxprops=dict(facecolor='lightsteelblue')
)
axes[1].set_title('Distribución de precio por cluster')
axes[1].set_ylabel('Precio por noche')

plt.tight_layout()
plt.show()
print(f'Varianza explicada PC1+PC2: {sum(pca.explained_variance_ratio_[:2]):.1%}')

In [ ]:
# Perfil normalizado por cluster
profile_vars = [c for c in cluster_features if c != 'price']
cluster_profile = df_clust.groupby('cluster')[profile_vars].mean()
cluster_profile_norm = (cluster_profile - cluster_profile.min()) / \
                       (cluster_profile.max() - cluster_profile.min())

cluster_profile_norm.T.plot(kind='bar', figsize=(12, 5), colormap='Set1')
plt.title('Perfil normalizado de cada cluster')
plt.xlabel('Variable')
plt.ylabel('Valor normalizado (0–1)')
plt.legend(title='Cluster', labels=labels)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Interpretación de los clusters:**

| Cluster | Descripción | Precio mediano | Capacidad media | Tamaño |
|---|---|---|---|---|
| **0 – Económico** | Propiedades pequeñas, mayor número de reseñas, alta rotación | $153 | 3.6 huéspedes | 46,910 (75%) |
| **1 – Intermedio/Premium** | Alojamientos más grandes, familias, mayor precio | $359 | 8.6 huéspedes | 15,644 (25%) |
| **2 – Ultra-premium/Outliers** | Solo 168 propiedades con precios extremos (~$50,000). Probablemente hoteles de lujo o errores de datos. | $50,000 | 5.8 huéspedes | 168 (<1%) |

**Conclusión:** El cluster 2 es claramente anómalo — propiedades con precios de $50,000/noche no corresponden al perfil típico de Airbnb y pueden distorsionar los modelos. Para las actividades de modelado se considerará si conviene filtrarlos.

---
## Actividad 4 – División Train / Test

In [ ]:
y = df_model['price']
X = df_model.drop(columns=['price'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print('=== Criterio de división ===')
print(f'  Proporción:   70% entrenamiento / 30% prueba')
print(f'  Filas totales: {len(df_model):,}')
print(f'  Train:         {len(X_train):,} filas')
print(f'  Test:          {len(X_test):,} filas')
print(f'  Features:      {X_train.shape[1]} columnas')
print(f'  Semilla:       42 (reproducibilidad)')
print()
print(pd.DataFrame({'Train': y_train.describe(), 'Test': y_test.describe()}).round(2))

In [ ]:
p95 = y.quantile(0.95)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(y_train[y_train <= p95], bins=50, alpha=0.6, label='Train', color='steelblue')
ax.hist(y_test[y_test <= p95], bins=50, alpha=0.6, label='Test', color='coral')
ax.set_title('Distribución de price: Train vs Test (sin outliers extremos)')
ax.set_xlabel('Precio por noche')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

**Criterio de división:**
- Se utilizó una proporción **70/30** (53,372 train / 22,874 test), que permite mayor variedad en el conjunto de prueba dado el volumen del dataset.
- La división **no se estratificó** ya que `price` es variable continua; la estratificación aplica para variables categóricas.
- Se fijó `random_state=42` para garantizar reproducibilidad.
- Las distribuciones de `price` en train y test son prácticamente idénticas (mediana $193 en ambos), lo que confirma que la partición es representativa y no introduce sesgo.

---
## Actividad 5 – Árbol de Regresión (todas las variables)

In [ ]:
tree_full = DecisionTreeRegressor(random_state=42)
tree_full.fit(X_train, y_train)

yp_train = tree_full.predict(X_train)
yp_test  = tree_full.predict(X_test)

rmse_tr = np.sqrt(mean_squared_error(y_train, yp_train))
rmse_te = np.sqrt(mean_squared_error(y_test, yp_test))
mae_tr  = mean_absolute_error(y_train, yp_train)
mae_te  = mean_absolute_error(y_test, yp_test)
r2_tr   = r2_score(y_train, yp_train)
r2_te   = r2_score(y_test, yp_test)

header = f"{'Metrica':<10} {'Train':>12} {'Test':>12}"
print(header)
print('-' * 36)
print(f"{'RMSE':<10} {rmse_tr:>12.2f} {rmse_te:>12.2f}")
print(f"{'MAE':<10} {mae_tr:>12.2f} {mae_te:>12.2f}")
print(f"{'R2':<10} {r2_tr:>12.4f} {r2_te:>12.4f}")
print(f"\nProfundidad: {tree_full.get_depth()} niveles")
print(f"Hojas:       {tree_full.get_n_leaves():,}")

In [ ]:
# Visualización del árbol (primeros 4 niveles)
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(tree_full, max_depth=4, filled=True,
          feature_names=X_train.columns.tolist(),
          rounded=True, ax=ax, fontsize=7)
ax.set_title('Árbol de Regresión – Primeros 4 niveles', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables y Real vs Predicho
importances = pd.Series(tree_full.feature_importances_, index=X_train.columns)
top15 = importances.nlargest(15).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top15.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 variables más importantes')
axes[0].set_xlabel('Importancia (reducción de impureza)')

p95 = y_test.quantile(0.95)
mask = y_test <= p95
axes[1].scatter(y_test[mask], yp_test[mask], alpha=0.3, s=8, color='steelblue')
axes[1].plot([0, p95], [0, p95], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_xlabel('Precio real')
axes[1].set_ylabel('Precio predicho')
axes[1].set_title('Real vs Predicho (sin outliers extremos)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Top 15 variables:')
print(top15.round(4).to_string())

**Análisis del árbol de regresión sin restricciones:**

| Métrica | Train | Test |
|---|---|---|
| RMSE | 108.18 | 1,721.09 |
| MAE | 2.01 | 202.42 |
| R² | 0.9993 | 0.8410 |

- **Overfitting severo:** El R² en train es 0.9993 (casi perfecto) mientras que en test cae a 0.8410. El árbol creció hasta 109 niveles con 54,358 hojas, memorizando el conjunto de entrenamiento en lugar de aprender patrones generalizables.
- **MAE en test ($202) vs train ($2):** El modelo predice con error promedio de $2 en train pero $202 en datos nuevos — evidencia clara de memorización.
- **Variables más importantes:** `room_type_Hotel room` (26.8%) domina, seguido por vecindarios específicos (Clearwater Beach, Gateway District) y métricas de disponibilidad. Esto indica que el árbol está aprendiendo identidades de anfitriones/vecindarios muy específicos en lugar de patrones generales.
- **Real vs Predicho:** El gráfico muestra buena predicción en precios bajos, pero los outliers de precio alto generan predicciones completamente fuera de rango.

**Conclusión:** Es necesario regularizar el árbol restringiendo la profundidad máxima y el número mínimo de muestras por hoja. Esto se abordará en la Actividad 7.